# 选修E1 · Day 2：Agent框架对比 · 上机练习（v5.0）

> **真实库**：LangGraph（真实运行）+ CrewAI/AutoGen（静态API对比）
> **核心对比**：ReAct vs Plan-Execute | LangGraph vs CrewAI vs AutoGen
> **营销映射**：同一个营销任务（透肌精华竞品分析+策略），不同框架实现对比

本笔记本包含 **6个TODO填空**，完成后你将：
1. 定义营销工具和离线StubLLM
2. 用`create_react_agent`构建LangGraph ReAct Agent
3. 用`StateGraph`构建LangGraph Plan-Execute Agent
4. 运行ReAct和Plan-Execute，对比步数/调用/输出
5. 用CrewAI API结构编写等价实现（静态对比设计哲学）
6. 用AutoGen API结构编写等价实现 + 生成四框架对比表

> 📦 真实库说明见 `data/README.md`
> 📖 理论讲义见 `notes.md`


In [ ]:
# === 导入真实库 ===
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage
from langchain_core.outputs import ChatResult, ChatGeneration
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# === 真实营销数据（基于护肤品电商场景，复用Day 1）===
PRODUCT_DB = {
    "透肌精华": "透肌焕亮精华液，299元，主打美白焕亮，含烟酰胺3%+维C衍生物，目标用户25-35岁都市白领。",
    "玻尿酸面霜": "玻尿酸保湿面霜，159元，主打深层补水，含双重玻尿酸，目标用户18-30岁女性。",
}
COMPETITOR_DB = {
    "雅诗兰黛": "雅诗兰黛小棕瓶精华，760元/30ml，市场占有率18%，优势：品牌力强、渠道完善；劣势：价格高、年轻化不足。",
    "兰蔻": "兰蔻小黑瓶精华，780元/30ml，市场占有率15%，优势：科技感强、专柜体验；劣势：下沉市场覆盖弱。",
}

# === 离线模拟LLM（无需API Key，预编排工具调用序列）===
class StubChatModel(BaseChatModel):
    """离线模拟LLM，预编排工具调用序列，保证无API Key可运行。
    替换为ChatOpenAI/ChatAnthropic即可使用真实LLM。"""
    responses: list = []
    call_index: int = 0

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        idx = self.call_index
        self.call_index += 1
        if idx < len(self.responses):
            resp = self.responses[idx]
        else:
            resp = AIMessage(content="任务完成。")
        return ChatResult(generations=[ChatGeneration(message=resp)])

    @property
    def _llm_type(self):
        return "stub"

    def bind_tools(self, tools, **kwargs):
        return self

# 统一的营销任务（所有框架用同一个任务对比）
MARKETING_TASK = "为透肌精华制定营销策略，竞品分析雅诗兰黛，并写入策略文件"

print("真实库导入成功")
print(f"  产品库: {list(PRODUCT_DB.keys())}")
print(f"  竞品库: {list(COMPETITOR_DB.keys())}")
print(f"  统一营销任务: {MARKETING_TASK}")
print(f"  StubChatModel: 离线模式（无API Key可运行）")
print(f"  crewai/autogen: 未安装 -> 采用静态API结构对比（不阻塞）")


---
## TODO1：用@tool装饰器定义营销工具

工具是Agent的"手"。在LangChain中，用`@tool`装饰器定义工具。
工具的**名称、docstring、参数类型**就是LLM看到的"接口契约"。

这些工具将被LangGraph的ReAct Agent和Plan-Execute Agent共用，
保证同一个营销任务在不同框架下的工具调用一致。

需要定义三个营销工具：
1. `search_product_info(product_name)` - 搜索产品信息（使用PRODUCT_DB）
2. `analyze_competitor(competitor_name)` - 分析竞品策略（使用COMPETITOR_DB）
3. `write_strategy(filename, content)` - 将策略写入文件

> 参考教材 § Day 1 四、工具使用（工具定义复用Day 1，保证任务一致性）


In [ ]:
# TODO1: 用@tool装饰器定义三个营销工具
# 提示: 使用 PRODUCT_DB 和 COMPETITOR_DB
# 工具1: search_product_info(product_name: str) -> str
# 工具2: analyze_competitor(competitor_name: str) -> str
# 工具3: write_strategy(filename: str, content: str) -> str
# 注意: docstring是LLM看到的接口契约，要写清楚！

# TODO: 你的代码
raise NotImplementedError

# 验证工具（取消注释后测试）
# tools = [search_product_info, analyze_competitor, write_strategy]
# print(f"定义了 {len(tools)} 个工具")
# result = search_product_info.invoke({"product_name": "透肌精华"})
# print(f"测试: {result}")


---
## TODO2：用create_react_agent构建LangGraph ReAct Agent

ReAct（Reasoning + Acting）的核心循环：
```
Thought -> Action -> Observation -> Thought -> ... -> FINISH
```

用LangGraph的`create_react_agent`构建ReAct Agent：
- model: 使用StubChatModel（预编排工具调用序列）
- tools: 使用TODO1定义的三个工具
- prompt: 系统提示，定义Agent角色

预编排轨迹模拟LLM的Thought-Action决策：搜索产品 -> 分析竞品 -> 写策略 -> 完成

> 参考教材 § Day 1 三、ReAct范式（本Day用同一范式做框架对比基准）


In [ ]:
# TODO2: 用create_react_agent构建LangGraph ReAct Agent
# 提示:
#   1. 创建 react_trajectory 列表，预编排4个AIMessage
#      - 前3个带 tool_calls (search_product_info, analyze_competitor, write_strategy)
#      - 最后一个带 content (最终回答)
#   2. 用 StubChatModel(responses=react_trajectory) 创建模型
#   3. 用 create_react_agent(model, tools, prompt=...) 构建Agent

# TODO: 你的代码
raise NotImplementedError

# 验证Agent（取消注释后测试）
# print(f"ReAct Agent构建成功，工具: {[t.name for t in tools]}")


---
## TODO3：用StateGraph构建LangGraph Plan-Execute Agent

Plan-Execute与ReAct的核心区别：
- **ReAct**：边推理边执行，每步可根据观测调整下一步
- **Plan-Execute**：先一次性规划所有步骤，再顺序执行

用LangGraph的`StateGraph`实现：
1. 定义`PlanExecuteState`（TypedDict）：task, plan, current_step, results, final_answer
2. `plan_node`：一次性生成完整计划（4步）
3. `execute_node`：根据当前步骤描述执行
4. `should_continue`：条件函数，判断是否还有未执行步骤
5. 构建图：plan -> execute -> (continue: execute | end: END)

> 参考教材 § Day 2 一、LangGraph设计哲学（Agent即图，显式控制流）


In [ ]:
# TODO3: 用StateGraph构建LangGraph Plan-Execute Agent
# 提示:
#   1. 定义 PlanExecuteState (TypedDict): task, plan, current_step, results, final_answer
#   2. plan_node: 生成4步计划（搜索产品/分析竞品/撰写策略/写入文件）
#   3. execute_node: 根据步骤描述执行，使用 PRODUCT_DB / COMPETITOR_DB
#   4. should_continue: current_step < len(plan) 则 continue，否则 end
#   5. 构建图: plan -> execute -> (continue: execute | end: END)
#   6. 编译图

# TODO: 你的代码
raise NotImplementedError

# 验证Agent（取消注释后测试）
# print(f"Plan-Execute Agent构建成功")


---
## TODO4：运行ReAct和Plan-Execute，对比执行轨迹

同一个营销任务，两种模式实跑，对比：
- 工具调用次数
- 模型调用次数（ReAct）/ 步骤数（Plan-Execute）
- 输出质量
- 执行模式差异

> 天道推演视角：ReAct的因果链在每步涌现，Plan-Execute的因果链在Plan阶段确定


In [ ]:
# TODO4: 运行ReAct和Plan-Execute，对比执行轨迹
# 提示:
#   1. 用 react_agent.invoke({"messages": [("user", MARKETING_TASK)]}) 运行ReAct
#   2. 用 plan_execute_agent.invoke({"task": "透肌精华"}) 运行Plan-Execute
#   3. 遍历ReAct的result["messages"]，统计tool_calls_count
#   4. 遍历Plan-Execute的plan和results，统计步数
#   5. 打印对比表

# TODO: 你的代码
raise NotImplementedError


---
## TODO5：用CrewAI API结构编写等价实现（静态对比）

CrewAI采用"Agent即角色"设计哲学：
- 开发者定义Agent（role/goal/backstory）和Task（description/expected_output/agent）
- CrewAI根据Task的context依赖自动编排执行顺序
- 角色化协作，代码简洁

> ⚠️ 本环境未安装crewai。按v5.0规则不pip install，采用**静态API结构对比**：
> 编写真实CrewAI API代码（可读性等同实跑），用try/except ImportError处理，
> 代码结构真实反映CrewAI设计哲学。

> 参考教材 § Day 2 一、CrewAI设计哲学 + 二、双框架实现


In [ ]:
# TODO5: 用CrewAI API结构编写等价实现（静态对比）
# 提示:
#   1. try: from crewai import Agent, Task, Crew, Process
#      except ImportError: 打印提示信息（不阻塞）
#   2. 定义4个角色化Agent:
#      - product_researcher (产品调研专家)
#      - pricing_analyst (定价策略分析师) [本任务可省略，聚焦3角色]
#      - marketing_analyst (营销活动分析师) [可省略]
#      - report_writer (竞品分析报告撰写人)
#      本任务简化为: product_researcher, competitor_analyst, report_writer
#   3. 定义3个Task，report_task的context依赖前两个task
#   4. 组建Crew，process=Process.sequential
#   5. 打印CrewAI设计哲学对比表

# TODO: 你的代码
raise NotImplementedError


---
## TODO6：用AutoGen API结构编写等价实现 + 四框架对比表

AutoGen采用"Agent即对话者"设计哲学：
- 每个Agent是ConversableAgent，可发送和接收消息
- 通过GroupChat机制，多个Agent在同一个对话中交互
- 适合需要Agent间讨论和协商的场景

> ⚠️ 本环境未安装autogen。采用**静态API结构对比**。

最后生成**四框架对比表**（LangGraph/CrewAI/AutoGen/MetaGPT），
作为本Day的核心交付物。

> 参考教材 § Day 2 三、AutoGen和MetaGPT的适用场景


In [ ]:
# TODO6: 用AutoGen API结构编写等价实现 + 四框架对比表
# 提示:
#   1. try: from autogen import ConversableAgent, GroupChat, GroupChatManager
#      except ImportError: 打印提示信息（不阻塞）
#   2. 编写AutoGen等价代码: 3个ConversableAgent + GroupChat + initiate_chat
#   3. 生成四框架对比表（LangGraph/CrewAI/AutoGen/MetaGPT）
#      维度: 设计哲学/核心抽象/控制流/灵活性/适用场景/本Day状态
#   4. 打印框架选择决策树

# TODO: 你的代码
raise NotImplementedError
